# Countersign pool — GPU steps (calibration → distances → metric validation)
Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
Everything is cached on Drive; if Colab disconnects, just Run all again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
DRIVE = '/content/drive/MyDrive/countersign'
assert os.path.exists(f'{DRIVE}/countersign_bundle.zip'), 'upload countersign_bundle.zip to MyDrive/countersign'
!rm -rf /content/cs && mkdir -p /content/cs && cd /content/cs && unzip -q -o {DRIVE}/countersign_bundle.zip
os.makedirs(f'{DRIVE}/countersign_runs', exist_ok=True)
# runs/ lives on Drive: caches and results survive disconnects
!cd /content/cs && cp -rn runs/. {DRIVE}/countersign_runs/ && rm -rf runs && ln -s {DRIVE}/countersign_runs runs
!ls /content/cs /content/cs/runs

In [ ]:
!pip -q install numba pyyaml openai-whisper edge-tts gTTS librosa soundfile statsmodels funasr modelscope kaldi-native-fbank
# stop Run all here if anything is missing
import whisper, funasr, edge_tts, gtts, numba, yaml, librosa, soundfile, statsmodels, kaldi_native_fbank
import torch; assert torch.cuda.is_available(), 'no GPU: Runtime -> Change runtime type -> T4 GPU'
print('OK, GPU:', torch.cuda.get_device_name(0))

In [ ]:
# trial/TTS cache on local disk (SQLite is unreliable on Drive), backed up to Drive every 5 min
os.environ['CS_CACHE'] = '/content/cache'
!mkdir -p /content/cache {DRIVE}/countersign_cache && cp -rn {DRIVE}/countersign_cache/. /content/cache/
subprocess.Popen(['bash', '-c', f'while true; do sleep 300; cp -r /content/cache/. {DRIVE}/countersign_cache/; done'])
def backup():
    subprocess.run(['bash', '-c', f'cp -r /content/cache/. {DRIVE}/countersign_cache/'])
def step(script):
    # a failing step raises, so Run all stops instead of running the next steps on missing files
    p = subprocess.Popen(['python', '-X', 'utf8', script], cwd='/content/cs', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=os.environ)
    for line in p.stdout:
        if 'it/s' not in line:
            print(line, end='', flush=True)
    p.wait()
    backup()
    if p.returncode:
        raise RuntimeError(f'{script} failed (exit {p.returncode}); see the traceback above')

In [ ]:
step('scripts/s2_calibration.py')

In [ ]:
step('scripts/s3_distances.py')

In [ ]:
step('scripts/s4_metric_validation.py')

In [ ]:
!ls -la /content/cs/runs/calibration /content/cs/runs/metric_validation
backup()
drive.flush_and_unmount()
print('flushed to Drive: safe to close')

Done. Results are in `MyDrive/countersign/countersign_runs/` (`calibration/`, `distances/summary.json`, `metric_validation/`).